# SQL 练习 2：做计算

上一课学了：`SELECT` 要哪些列、`WHERE` 筛行、`COUNT(*)` 数行、`GROUP BY` 分组、`ORDER BY` 排序。

这一课只学一件事：**除了数行数，还能算总和、平均、最大、最小**。

运行方法：点中格子按 **Shift+Enter**。右上角内核要选 **.venv**。

In [ ]:
import duckdb

con = duckdb.connect(r"C:\Users\user\Desktop\AI analysis\Data\aiusage.duckdb", read_only=True)
print("连上了")

## 第 1 条：求总和 `SUM`

`COUNT(*)` 数的是「有多少行」，`SUM(列)` 算的是「这一列加起来是多少」。

下面这条在问：所有对话加起来，我一共提问了多少次？

**结果应该是 12323**

In [ ]:
con.sql("SELECT SUM(user_turns) AS total_prompts FROM sessions").df()

## 第 2 条：平均、最大、最小

- `AVG(列)` 平均值
- `MAX(列)` 最大值
- `MIN(列)` 最小值
- `ROUND(值, 1)` 保留一位小数（不然平均值会是一长串）

**一次可以算好几个**，用逗号隔开就行。

**结果应该是：平均 11.7 次、最多 408 次、最少 1 次**

In [ ]:
con.sql("""
    SELECT
        ROUND(AVG(user_turns), 1) AS avg_prompts,
        MAX(user_turns)           AS most,
        MIN(user_turns)           AS least
    FROM sessions
""").df()

上面这几个（`COUNT`、`SUM`、`AVG`、`MAX`、`MIN`）叫**聚合函数**：
特点是把**很多行压成一个值**。所以结果只有一行。

## 第 3 条：分组之后再算

上一条算的是全部对话的平均。加上 `GROUP BY platform`，就变成**每个平台分别算一遍**。

聚合函数可以和 `GROUP BY` 任意组合，这是 SQL 里最常用的搭配。

**结果应该是这样（按提问总数排序）：**

| platform | sessions | prompts | avg_prompts |
|---|---|---|---|
| chatgpt | 681 | 7500 | 11.0 |
| gemini | 170 | 2773 | 16.3 |
| claude_code | 64 | 1056 | 16.5 |
| claude_web | 75 | 592 | 7.9 |
| codex | 64 | 402 | 6.3 |

In [ ]:
con.sql("""
    SELECT
        platform,
        COUNT(*)                  AS sessions,
        SUM(user_turns)           AS prompts,
        ROUND(AVG(user_turns), 1) AS avg_prompts
    FROM sessions
    GROUP BY platform
    ORDER BY prompts DESC
""").df()

**读一下这个结果**：ChatGPT 的对话数量最多（681 个），但平均每个对话只提问 11 次；
Gemini 和 Claude Code 的对话少，但聊得更深（平均 16 次）。

## 第 4 条：先筛行，再分组

`WHERE` 放在 `GROUP BY` 前面，表示**先把不要的行扔掉，再分组计算**。

下面只看网页版（`source = 'web'`），按哪家 AI 分组。

**结果应该是：chatgpt 7500 次（平均 11.0）、gemini 2773 次（平均 16.3）、claude 592 次（平均 7.9）**

In [ ]:
con.sql("""
    SELECT
        session_ai,
        SUM(user_turns)           AS prompts,
        ROUND(AVG(user_turns), 1) AS avg_prompts
    FROM sessions
    WHERE source = 'web'
    GROUP BY session_ai
    ORDER BY prompts DESC
""").df()

## 第 5 条：筛选「算完之后」的结果 `HAVING`

假如只想看**平均提问数超过 12 次**的平台。这个条件是算完平均才知道的，`WHERE` 用不了（它在分组之前就执行了）。

这种情况用 **`HAVING`**：它在分组算完之后再筛。

| 关键字 | 什么时候执行 | 筛什么 |
|---|---|---|
| `WHERE` | 分组**之前** | 原始的行 |
| `HAVING` | 分组**之后** | 每组算出来的结果 |

**结果应该只剩两行：claude_code 16.5、gemini 16.3**

In [ ]:
con.sql("""
    SELECT
        platform,
        ROUND(AVG(user_turns), 1) AS avg_prompts
    FROM sessions
    GROUP BY platform
    HAVING AVG(user_turns) > 12
    ORDER BY avg_prompts DESC
""").df()

## 完整的执行顺序

到这里，一条 SQL 的六个部分都学过了。**理解时按这个顺序想**：

| 顺序 | 关键字 | 做什么 |
|---|---|---|
| 1 | `FROM` | 拿出整张表 |
| 2 | `WHERE` | 扔掉不要的行 |
| 3 | `GROUP BY` | 把剩下的行分成几堆 |
| 4 | `SELECT` | 每堆算出一行（聚合函数在这一步） |
| 5 | `HAVING` | 扔掉不要的结果行 |
| 6 | `ORDER BY` / `LIMIT` | 排序、只留前几行 |

写的时候顺序是 `SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT`。

## 自己试试

1. 我一共向 AI 输入了多少 token？（`SUM(user_tokens)`）
2. 每个平台的 `tool_tokens` 总和是多少？哪两个平台是 0？
3. 只看 `session_type = 'coding'` 的对话，平均每个对话有多少次提问？

答案在下一个格子的注释里。

In [ ]:
# 在这里写你的 SQL


# 答案：
# 1. 300,787 个 token（AI 回复了 6,736,100 个，是你输入的 22 倍）
# 2. claude_code 15,821,646；codex 14,812,155；claude_web 763,549；chatgpt 和 gemini 都是 0
#    （网页版看不到工具调用，所以是 0）
# 3. 11.4 次